In [ ]:
# # GMM Analysis — Influenza Segment Distance Distributions
# 
# Loads and visualises outputs from:
# - `gmm_per_segment.sh` (per-segment 1D GMMs with BIC/AIC/ICL/modBIC)
# - `gmm_joint_8d.sh` (joint 8D GMM across all segments)


In [ ]:
#── Imports & paths ────────────────────────────────────────
import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from scipy.stats import norm
from pathlib import Path

SEGMENT_NAMES = ["PB2", "PB1", "PA", "HA", "NP", "NA", "MP", "NS"]

# ── EDIT THESE PATHS ──
BASE_DIR = Path("/data/users/ltucker/influenzaData/H5N1_pipeline/output")
GMM_DIR = BASE_DIR / "famsa_per_segment_analysis" / "gmm"
JOINT_DIR = BASE_DIR / "famsa_per_segment_analysis" / "gmm_joint_8d"
DIST_DIR = BASE_DIR / "famsa_distances"

print(f"GMM dir:   {GMM_DIR}")
print(f"Joint dir: {JOINT_DIR}")
print(f"Dist dir:  {DIST_DIR}")


In [ ]:
#── Load per-segment results ───────────────────────────────
seg_data = {}
for seg_name in SEGMENT_NAMES:
    jpath = GMM_DIR / f"gmm_{seg_name}.json"
    if jpath.exists():
        with open(jpath) as f:
            seg_data[seg_name] = json.load(f)
        print(f"✓ {seg_name}: primary_k={seg_data[seg_name]['primary_k']}")
    else:
        print(f"✗ {seg_name}: not found")

# Combined criteria table
criteria_dfs = []
for seg_name in seg_data:
    cpath = GMM_DIR / f"gmm_criteria_{seg_name}.csv"
    if cpath.exists():
        criteria_dfs.append(pd.read_csv(cpath))
if criteria_dfs:
    criteria_all = pd.concat(criteria_dfs, ignore_index=True)
    print(f"\nCombined criteria table: {len(criteria_all)} rows")
    display(criteria_all.head(12))


In [ ]:
#── 1. Best-k heatmap across segments & criteria ───────────
criteria_names = ["bic", "aic", "icl", "mod_bic"]
labels_pretty = ["BIC", "AIC", "ICL", "Modified BIC"]

best_k_matrix = []
for seg_name in SEGMENT_NAMES:
    if seg_name in seg_data:
        row = [seg_data[seg_name]["best_k_per_criterion"].get(c, np.nan) for c in criteria_names]
    else:
        row = [np.nan] * 4
    best_k_matrix.append(row)

best_k_df = pd.DataFrame(best_k_matrix, index=SEGMENT_NAMES, columns=labels_pretty)

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(best_k_df.values, cmap="YlOrRd", aspect="auto", vmin=2, vmax=7)
ax.set_xticks(range(len(labels_pretty)))
ax.set_xticklabels(labels_pretty, fontsize=11)
ax.set_yticks(range(len(SEGMENT_NAMES)))
ax.set_yticklabels(SEGMENT_NAMES, fontsize=11)
for i in range(len(SEGMENT_NAMES)):
    for j in range(len(labels_pretty)):
        v = best_k_df.iloc[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{int(v)}", ha="center", va="center", fontsize=14, fontweight="bold",
                    color="white" if v >= 5 else "black")
plt.colorbar(im, label="Best k", shrink=0.8)
ax.set_title("Best k per segment per criterion", fontsize=13, weight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# ── 2. Criteria curves (normalised) — all segments overlaid 
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
seg_colors = plt.cm.tab10(np.linspace(0, 1, len(SEGMENT_NAMES)))

for ax, (col, label) in zip(axes, zip(criteria_names, labels_pretty)):
    for i, seg_name in enumerate(SEGMENT_NAMES):
        if seg_name not in seg_data:
            continue
        df_seg = criteria_all[criteria_all["segment"] == seg_name]
        if df_seg.empty or col not in df_seg.columns:
            continue
        vals = df_seg[col].values
        if len(vals) == 0:
            continue
        vals_n = (vals - vals.min()) / (vals.max() - vals.min() + 1e-10)
        ax.plot(df_seg["k"].values, vals_n, "o-", color=seg_colors[i],
                linewidth=2, markersize=6, label=seg_name)
        best_k = seg_data[seg_name]["best_k_per_criterion"].get(col)
        if best_k and best_k in df_seg["k"].values:
            idx = df_seg["k"].values.tolist().index(best_k)
            ax.plot(best_k, vals_n[idx], "*", color=seg_colors[i],
                    markersize=14, markeredgecolor="black", markeredgewidth=0.5)
    ax.set_xlabel("k")
    ax.set_ylabel(f"{label} (normalised)")
    ax.set_title(label)
    ax.set_xticks(range(2, 8))
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7, ncol=2)

plt.suptitle("Model selection criteria across segments (★ = best k)", fontsize=14, weight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── 3. Per-segment raw criteria (absolute values)
fig, axes = plt.subplots(2, 4, figsize=(22, 8))
axes = axes.flatten()

for i, seg_name in enumerate(SEGMENT_NAMES):
    ax = axes[i]
    if seg_name not in seg_data:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(seg_name)
        continue
    df_seg = criteria_all[criteria_all["segment"] == seg_name]
    if df_seg.empty:
        ax.text(0.5, 0.5, "No criteria data", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(seg_name)
        continue
    for col, label, color in zip(criteria_names, labels_pretty,
                                  ["steelblue", "darkorange", "forestgreen", "firebrick"]):
        vals = df_seg[col].values
        if len(vals) == 0:
            continue
        vals_n = (vals - vals.min()) / (vals.max() - vals.min() + 1e-10)
        ax.plot(df_seg["k"].values, vals_n, "o-", color=color, linewidth=2, label=label)
    ax.set_title(seg_name, fontsize=12, weight="bold")
    ax.set_xticks(range(2, 8))
    ax.set_xlabel("k")
    ax.grid(True, alpha=0.3)
    if i == 0:
        ax.legend(fontsize=7)
        ax.set_ylabel("Normalised score")

plt.suptitle("Per-segment criteria (each normalised within its own range)", fontsize=14, weight="bold")
plt.tight_layout()
plt.show()

In [ ]:
#── 4. Reconstruct & plot GMM density overlays from JSON ───
def load_distance_vector(seg_idx):
    """Load upper triangle from symmetric distance parquet."""
    fpath = DIST_DIR / f"symmetric_distances_{seg_idx + 1}.parquet"
    df = pd.read_parquet(fpath)
    mat = df.iloc[:, 1:].values.astype(float)
    n = min(mat.shape[0], mat.shape[1])
    mat = mat[:n, :n]
    return mat[np.triu_indices(n, k=1)]

fig, axes = plt.subplots(2, 4, figsize=(24, 10))
axes = axes.flatten()

for seg_idx, seg_name in enumerate(SEGMENT_NAMES):
    ax = axes[seg_idx]
    if seg_name not in seg_data:
        ax.set_title(f"{seg_name} — no data")
        continue

    upper_tri = load_distance_vector(seg_idx)
    sd = seg_data[seg_name]
    primary_k = sd["primary_k"]

    # Find the iteration matching primary_k and use full-data params
    it = next(r for r in sd["iterations"] if r["k"] == primary_k)
    weights = it["full_data_weights"]
    means = it["full_data_means"]
    covs = it["full_data_covariances"]

    x = np.linspace(upper_tri.min(), upper_tri.max(), 1000)
    colors_k = plt.cm.tab10(np.arange(primary_k))

    ax.hist(upper_tri, bins=150, density=True, alpha=0.3, color="gray", edgecolor="none")
    for c in range(primary_k):
        sigma = np.sqrt(covs[c])
        curve = weights[c] * norm.pdf(x, means[c], sigma)
        ax.plot(x, curve, color=colors_k[c], linewidth=2,
                label=f"C{c+1} (μ={means[c]:.3f}, w={weights[c]:.2f})")
    ax.set_title(f"{seg_name} (k={primary_k})", fontsize=12, weight="bold")
    ax.set_xlabel("Distance")
    if seg_idx % 4 == 0:
        ax.set_ylabel("Density")
    ax.legend(fontsize=6, loc="upper right")

plt.suptitle("GMM density decomposition — all 8 segments (full-data fit, ICL-selected k)",
             fontsize=14, weight="bold")
plt.tight_layout()
plt.show()


In [ ]:
#── 5. All k fits for each segment (grid) ─────────────────
for seg_idx, seg_name in enumerate(SEGMENT_NAMES):
    if seg_name not in seg_data:
        continue
    sd = seg_data[seg_name]
    upper_tri = load_distance_vector(seg_idx)
    x = np.linspace(upper_tri.min(), upper_tri.max(), 1000)

    n_k = len(sd["iterations"])
    fig, axes_k = plt.subplots(1, n_k, figsize=(4.5 * n_k, 4), sharey=True)
    if n_k == 1:
        axes_k = [axes_k]

    for i_ax, it in enumerate(sd["iterations"]):
        ax = axes_k[i_ax]
        k = it["k"]
        weights = it["full_data_weights"]
        means = it["full_data_means"]
        covs = it["full_data_covariances"]
        ck = plt.cm.tab10(np.arange(k))

        ax.hist(upper_tri, bins=150, density=True, alpha=0.3, color="gray", edgecolor="none")
        for c in range(k):
            sigma = np.sqrt(covs[c])
            curve = weights[c] * norm.pdf(x, means[c], sigma)
            ax.plot(x, curve, color=ck[c], linewidth=1.5)

        # Mark which criteria selected this k
        selected = [name for name, bk in sd["best_k_per_criterion"].items() if bk == k]
        tag = f"  ← {', '.join(selected)}" if selected else ""
        ax.set_title(f"k={k}{tag}", fontsize=10)
        ax.set_xlabel("Distance")
        if i_ax == 0:
            ax.set_ylabel("Density")

    fig.suptitle(f"{seg_name} — all k fits", fontsize=13, weight="bold")
    plt.tight_layout()
    plt.show()


In [ ]:
#── 6. Component means & weights scatter ──────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for seg_idx, seg_name in enumerate(SEGMENT_NAMES):
    if seg_name not in seg_data:
        continue
    sd = seg_data[seg_name]
    it = next(r for r in sd["iterations"] if r["k"] == sd["primary_k"])
    means = it["full_data_means"]
    weights = it["full_data_weights"]
    k = it["k"]

    # Sort by mean
    order = np.argsort(means)
    means_s = [means[o] for o in order]
    weights_s = [weights[o] for o in order]

    colors_k = plt.cm.tab10(np.arange(k))
    axes[0].scatter([seg_idx] * k, means_s, s=[w * 600 for w in weights_s],
                    c=colors_k, edgecolors="black", linewidth=0.5, zorder=3)
    for m, w in zip(means_s, weights_s):
        axes[0].annotate(f"{w:.2f}", (seg_idx + 0.15, m), fontsize=7, color="gray")

axes[0].set_xticks(range(len(SEGMENT_NAMES)))
axes[0].set_xticklabels(SEGMENT_NAMES)
axes[0].set_ylabel("Component mean (distance)")
axes[0].set_title("Component means per segment (size ∝ weight)", fontsize=12)
axes[0].grid(True, alpha=0.3, axis="y")

# Bar chart of selected k
ks = [seg_data.get(s, {}).get("primary_k", 0) for s in SEGMENT_NAMES]
bar_colors = plt.cm.Set2(np.linspace(0, 1, len(SEGMENT_NAMES)))
bars = axes[1].bar(range(len(SEGMENT_NAMES)), ks, color=bar_colors, edgecolor="black")
for bar, k_val in zip(bars, ks):
    if k_val > 0:
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                     str(k_val), ha="center", fontsize=12, fontweight="bold")
axes[1].set_xticks(range(len(SEGMENT_NAMES)))
axes[1].set_xticklabels(SEGMENT_NAMES)
axes[1].set_ylabel("Number of components (k)")
axes[1].set_title("ICL-selected k per segment", fontsize=12)
axes[1].set_yticks(range(0, 8))
axes[1].grid(True, alpha=0.3, axis="y")

plt.suptitle("Cross-segment GMM summary", fontsize=14, weight="bold")
plt.tight_layout()
plt.show()


In [ ]:
#── 7. Subsample vs full-data parameter comparison ─────────
# Check if full-data means differ from subsample means
fig, axes = plt.subplots(2, 4, figsize=(22, 8))
axes = axes.flatten()

for seg_idx, seg_name in enumerate(SEGMENT_NAMES):
    ax = axes[seg_idx]
    if seg_name not in seg_data:
        ax.set_title(f"{seg_name} — no data")
        continue
    sd = seg_data[seg_name]
    for it in sd["iterations"]:
        k = it["k"]
        sub_means = sorted(it["means"])
        full_means = sorted(it["full_data_means"])
        ax.plot(sub_means, full_means, "o", markersize=8, label=f"k={k}")
    # Plot y=x line
    lims = ax.get_xlim()
    ax.plot(lims, lims, "k--", alpha=0.3, linewidth=1)
    ax.set_xlabel("Subsample mean")
    ax.set_ylabel("Full-data mean")
    ax.set_title(seg_name, fontsize=11, weight="bold")
    ax.set_aspect("equal")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle("Subsample vs full-data component means (should cluster near diagonal)",
             fontsize=13, weight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# %% ── Map sample headers to per-segment GMM components ───────
from sklearn.mixture import GaussianMixture

def load_segment_pairs(seg_idx):
    """Load distance matrix with sample names, return pairs with distances."""
    # Sample names from original distance file
    name_file = DIST_DIR / f"distance_{seg_idx + 1}.parquet"
    df_names = pd.read_parquet(name_file)
    sample_names = df_names.iloc[:, 0].astype(str).tolist()

    # Numeric matrix from symmetric file
    dist_file = DIST_DIR / f"symmetric_distances_{seg_idx + 1}.parquet"
    df_dist = pd.read_parquet(dist_file)
    mat = df_dist.iloc[:, 1:].values.astype(float)
    n = min(mat.shape[0], mat.shape[1])
    mat = mat[:n, :n]
    sample_names = sample_names[:n]

    triu = np.triu_indices(n, k=1)
    distances = mat[triu]

    # Filter to [0, 0.1]
    mask = (distances >= 0) & (distances <= 0.1)
    return (
        [sample_names[i] for i, m in zip(triu[0], mask) if m],
        [sample_names[j] for j, m in zip(triu[1], mask) if m],
        distances[mask],
    )


# %% ── Build pair-level assignment table for each segment ─────
all_pair_tables = {}

for seg_idx, seg_name in enumerate(SEGMENT_NAMES):
    if seg_name not in seg_data:
        continue
    sd = seg_data[seg_name]
    primary_k = sd["primary_k"]
    it = next(r for r in sd["iterations"] if r["k"] == primary_k)

    # Reconstruct the GMM from saved full-data parameters
    gmm = GaussianMixture(n_components=primary_k, covariance_type="full")
    gmm.weights_ = np.array(it["full_data_weights"])
    gmm.means_ = np.array(it["full_data_means"]).reshape(-1, 1)
    gmm.covariances_ = np.array(it["full_data_covariances"]).reshape(-1, 1, 1)
    gmm.precisions_cholesky_ = np.linalg.cholesky(
        np.linalg.inv(gmm.covariances_)
    )

    # Load pairs with headers
    names_i, names_j, dists = load_segment_pairs(seg_idx)
    labels = gmm.predict(dists.reshape(-1, 1))

    pair_df = pd.DataFrame({
        "sample_i": names_i,
        "sample_j": names_j,
        "distance": dists,
        "component": labels,
    })
    all_pair_tables[seg_name] = pair_df
    print(f"{seg_name} (k={primary_k}): {len(pair_df):,} pairs")
    print(pair_df.groupby("component")["distance"].describe()[["count", "mean", "std"]].round(4))
    print()


# %% ── Explore: which samples dominate each component? ────────
seg_name = "HA"  # ← change this to explore other segments
pair_df = all_pair_tables[seg_name]

# Count how many times each sample appears in each component
sample_counts = []
for comp in sorted(pair_df["component"].unique()):
    comp_df = pair_df[pair_df["component"] == comp]
    all_samples = pd.concat([comp_df["sample_i"], comp_df["sample_j"]])
    counts = all_samples.value_counts().rename(f"component_{comp}")
    sample_counts.append(counts)

profile = pd.concat(sample_counts, axis=1).fillna(0).astype(int)
profile["total"] = profile.sum(axis=1)
profile["dominant_component"] = profile.drop(columns="total").idxmax(axis=1)
profile = profile.sort_values("total", ascending=False)

print(f"\n{seg_name}: top 10 samples by total pair count")
display(profile.head(10))


# %% ── Parse metadata from headers and add to profiles ────────
def parse_header(name):
    parts = name.split("|")
    if len(parts) >= 8:
        return {
            "subtype": parts[0],
            "isolate": parts[1],
            "segment": parts[2],
            "accession": parts[4],
            "country": parts[5],
            "continent": parts[6],
            "year": parts[7],
        }
    return {"isolate": name}

meta = pd.DataFrame([parse_header(n) for n in profile.index], index=profile.index)
profile_meta = pd.concat([profile, meta], axis=1)

print(f"\n{seg_name}: sample profiles with metadata")
display(profile_meta.head(10))


# %% ── Component composition by country/year/continent ────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
primary_k = seg_data[seg_name]["primary_k"]
cluster_colors = plt.cm.tab10(np.arange(primary_k))

for ax, field in zip(axes, ["continent", "country", "year"]):
    if field not in profile_meta.columns:
        continue

    # For each metadata value, compute the average proportion in each component
    comp_cols = [c for c in profile_meta.columns if c.startswith("component_")]
    grouped = profile_meta.groupby(field)[comp_cols].sum()
    proportions = grouped.div(grouped.sum(axis=1), axis=0)

    # Show top 10 categories by total samples
    top = grouped.sum(axis=1).nlargest(10).index
    proportions = proportions.loc[top]

    proportions.plot.barh(stacked=True, ax=ax, color=cluster_colors, edgecolor="none")
    ax.set_xlabel("Proportion of pairs")
    ax.set_title(f"Component breakdown by {field}", fontsize=11)
    ax.legend(fontsize=7, loc="lower right")

plt.suptitle(f"{seg_name}: GMM component composition by metadata", fontsize=14, weight="bold")
plt.tight_layout()
plt.show()


# %% ── Pairs of interest: find potential reassortant pairs ────
# Pairs that are in the CLOSE component for one segment but DISTANT in another
if len(all_pair_tables) >= 2:
    seg_a, seg_b = "HA", "PB1"  # ← change these
    df_a = all_pair_tables[seg_a].set_index(["sample_i", "sample_j"])
    df_b = all_pair_tables[seg_b].set_index(["sample_i", "sample_j"])

    # Join on shared pairs
    joined = df_a[["distance", "component"]].join(
        df_b[["distance", "component"]],
        lsuffix=f"_{seg_a}", rsuffix=f"_{seg_b}",
        how="inner"
    )
    print(f"Shared pairs between {seg_a} and {seg_b}: {len(joined):,}")

    # Discordant pairs: close in one segment, distant in another
    # Component 0 is typically the lowest-mean (closest) component
    discordant = joined[
        joined[f"component_{seg_a}"] != joined[f"component_{seg_b}"]
    ]
    #print(f"Discordant pairs (different component): {len(discordant):,} ({len(discordant)/len(joined)*100:.1f}%)")
    display(discordant.sort_values(f"distance_{seg_a}").head(10))

In [ ]:
[markdown]
# ---
# # Joint 8D GMM Analysis
# 
# The following cells load outputs from `gmm_joint_8d.sh`.


In [ ]:
#── Load joint 8D results ─────────────────────────────────
joint_json_path = JOINT_DIR / "gmm_joint_8d.json"
if joint_json_path.exists():
    with open(joint_json_path) as f:
        joint_data = json.load(f)
    print(f"Joint 8D GMM loaded:")
    print(f"  Shared samples: {joint_data['n_shared_samples']}")
    print(f"  Total pairs:    {joint_data['n_pairs_total']:,}")
    print(f"  Primary k:      {joint_data['primary_k']} (by {joint_data['primary_criterion']})")
    print(f"  Best k per criterion: {joint_data['best_k_per_criterion']}")
else:
    joint_data = None
    print(f"Joint 8D JSON not found at {joint_json_path}")
    print("Run gmm_joint_8d.sh first, then re-run these cells.")


In [ ]:
#── 8. Joint 8D criteria curves ───────────────────────────
if joint_data:
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.flatten()
    criterion_info = [
        ("bic", "BIC", "steelblue"),
        ("aic", "AIC", "darkorange"),
        ("icl", "ICL", "forestgreen"),
        ("mod_bic", "Modified BIC", "firebrick"),
    ]
    ks = [r["k"] for r in joint_data["iterations"]]
    best_kpc = joint_data["best_k_per_criterion"]

    for ax, (col, label, color) in zip(axes, criterion_info):
        vals = [r[col] for r in joint_data["iterations"]]
        ax.plot(ks, vals, "o-", color=color, linewidth=2, markersize=8)
        bk = best_kpc[col]
        ax.axvline(bk, color="red", linestyle="--", alpha=0.7, label=f"Best k={bk}")
        ax.set_xlabel("k")
        ax.set_ylabel(label)
        ax.set_title(f"Joint 8D — {label}")
        ax.set_xticks(ks)
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.suptitle("Joint 8D GMM: model selection criteria", fontsize=14, weight="bold")
    plt.tight_layout()
    plt.show()


In [ ]:
#── 9. Joint 8D cluster mean profiles (bar chart) ─────────
if joint_data:
    primary_k = joint_data["primary_k"]
    # Get the full-data means for the best k
    it = next(r for r in joint_data["iterations"] if r["k"] == primary_k)
    means_8d = np.array(it["full_data_means"])  # shape (k, 8)
    weights_8d = it["full_data_weights"]

    # Unscale back to raw distances
    scaler_means = np.array(joint_data["scaler"]["means"])
    scaler_stds = np.array(joint_data["scaler"]["stds"])
    means_raw = means_8d * scaler_stds + scaler_means

    fig, axes = plt.subplots(1, primary_k, figsize=(5 * primary_k, 5), sharey=True)
    if primary_k == 1:
        axes = [axes]

    cluster_colors = plt.cm.tab10(np.arange(primary_k))
    x_pos = np.arange(len(SEGMENT_NAMES))

    for c in range(primary_k):
        ax = axes[c]
        ax.bar(x_pos, means_raw[c], color=cluster_colors[c], edgecolor="black", alpha=0.7)
        ax.set_xticks(x_pos)
        ax.set_xticklabels(SEGMENT_NAMES, rotation=45, ha="right")
        ax.set_title(f"Cluster {c} (w={weights_8d[c]:.2f})", fontsize=11)
        if c == 0:
            ax.set_ylabel("Mean distance (raw)")
        ax.grid(True, alpha=0.3, axis="y")

    plt.suptitle(f"Joint 8D GMM (k={primary_k}): mean distance profile per cluster",
                 fontsize=14, weight="bold")
    plt.tight_layout()
    plt.show()


In [ ]:
# ── 10. Joint 8D cluster profiles — heatmap ───────────────
if joint_data:
    primary_k = joint_data["primary_k"]
    it = next(r for r in joint_data["iterations"] if r["k"] == primary_k)
    means_8d = np.array(it["full_data_means"])
    scaler_means = np.array(joint_data["scaler"]["means"])
    scaler_stds = np.array(joint_data["scaler"]["stds"])
    means_raw = means_8d * scaler_stds + scaler_means

    fig, ax = plt.subplots(figsize=(10, max(3, primary_k * 0.8 + 1)))
    im = ax.imshow(means_raw, aspect="auto", cmap="viridis")
    ax.set_xticks(range(len(SEGMENT_NAMES)))
    ax.set_xticklabels(SEGMENT_NAMES, fontsize=11)
    ax.set_yticks(range(primary_k))
    ax.set_yticklabels([f"Cluster {c} (w={it['full_data_weights'][c]:.2f})" for c in range(primary_k)],
                       fontsize=10)
    for i in range(primary_k):
        for j in range(len(SEGMENT_NAMES)):
            ax.text(j, i, f"{means_raw[i, j]:.3f}", ha="center", va="center",
                    fontsize=9, color="white" if means_raw[i, j] > means_raw.mean() else "black")
    plt.colorbar(im, label="Mean distance", shrink=0.8)
    ax.set_title(f"Joint 8D GMM (k={primary_k}): cluster × segment mean distances", fontsize=13, weight="bold")
    plt.tight_layout()
    plt.show()


In [ ]:
# ── 11. Sample cluster profiles ───────────────────────────
profile_path = JOINT_DIR / "sample_cluster_profiles.csv"
if profile_path.exists():
    profiles = pd.read_csv(profile_path, index_col=0)
    print(f"Sample profiles: {profiles.shape}")

    # Normalise each row to proportions
    profiles_norm = profiles.div(profiles.sum(axis=1), axis=0)

    fig, ax = plt.subplots(figsize=(12, 5))
    primary_k = profiles.shape[1]
    cluster_colors = plt.cm.tab10(np.arange(primary_k))
    bottom = np.zeros(len(profiles_norm))

    for c in range(primary_k):
        col = profiles_norm.columns[c]
        ax.bar(range(len(profiles_norm)), profiles_norm[col].values,
               bottom=bottom, color=cluster_colors[c], label=col, width=1.0, edgecolor="none")
        bottom += profiles_norm[col].values

    ax.set_xlabel("Sample index")
    ax.set_ylabel("Proportion of pairs in each cluster")
    ax.set_title("Per-sample cluster membership (stacked proportions)", fontsize=13, weight="bold")
    ax.legend(fontsize=9)
    ax.set_xlim(-0.5, len(profiles_norm) - 0.5)
    plt.tight_layout()
    plt.show()

    # Distribution of dominant cluster
    dominant = profiles_norm.idxmax(axis=1)
    fig, ax = plt.subplots(figsize=(8, 4))
    dominant.value_counts().sort_index().plot.bar(ax=ax, color=cluster_colors[:primary_k], edgecolor="black")
    ax.set_xlabel("Dominant cluster")
    ax.set_ylabel("Number of samples")
    ax.set_title("How many samples are dominated by each cluster", fontsize=12)
    plt.tight_layout()
    plt.show()


In [ ]:
# ── 12. Pair assignments — per-segment marginals by cluster 
pair_path = JOINT_DIR / "pair_assignments.parquet"
if pair_path.exists():
    print("Loading pair assignments (may take a moment)...")
    pairs = pd.read_parquet(pair_path)
    print(f"Pair assignments: {len(pairs):,} rows")
    primary_k = pairs["cluster"].nunique()

    fig, axes = plt.subplots(2, 4, figsize=(24, 10))
    axes = axes.flatten()
    cluster_colors = plt.cm.tab10(np.arange(primary_k))

    for seg_idx, seg_name in enumerate(SEGMENT_NAMES):
        ax = axes[seg_idx]
        col = f"dist_{seg_name}"
        if col not in pairs.columns:
            ax.set_title(f"{seg_name} — no data")
            continue
        for c in range(primary_k):
            mask = pairs["cluster"] == c
            ax.hist(pairs.loc[mask, col], bins=100, alpha=0.5,
                    color=cluster_colors[c], edgecolor="none",
                    label=f"Cluster {c} (n={mask.sum():,})", density=True)
        ax.set_title(seg_name, fontsize=12, weight="bold")
        ax.set_xlabel("Distance (raw)")
        if seg_idx % 4 == 0:
            ax.set_ylabel("Density")
        if seg_idx == 0:
            ax.legend(fontsize=7)

    plt.suptitle(f"Joint 8D GMM (k={primary_k}): per-segment marginals by cluster",
                 fontsize=14, weight="bold")
    plt.tight_layout()
    plt.show()


In [ ]:
# ── 13. Pair assignments — PCA scatter ─────────────────────
if pair_path.exists():
    from sklearn.decomposition import PCA

    dist_cols = [f"dist_{s}" for s in SEGMENT_NAMES]
    feature_matrix = pairs[dist_cols].values
    labels = pairs["cluster"].values

    # Subsample for plotting speed
    np.random.seed(42)
    vis_n = min(200_000, len(pairs))
    vis_idx = np.random.choice(len(pairs), size=vis_n, replace=False)

    pca = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(feature_matrix[vis_idx])
    vis_labels = labels[vis_idx]

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    # Scatter colored by cluster
    for c in range(primary_k):
        mask = vis_labels == c
        axes[0].scatter(coords[mask, 0], coords[mask, 1], s=0.5, alpha=0.2,
                        color=cluster_colors[c], label=f"Cluster {c}")
    axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
    axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
    axes[0].set_title("PCA of 8D pair distances (colored by cluster)")
    axes[0].legend(markerscale=20, fontsize=9)

    # PCA loadings biplot
    loadings = pca.components_.T
    for seg_idx, seg_name in enumerate(SEGMENT_NAMES):
        axes[1].arrow(0, 0, loadings[seg_idx, 0] * 3, loadings[seg_idx, 1] * 3,
                      head_width=0.05, head_length=0.02, fc="steelblue", ec="steelblue")
        axes[1].text(loadings[seg_idx, 0] * 3.3, loadings[seg_idx, 1] * 3.3,
                     seg_name, fontsize=11, ha="center", weight="bold")
    axes[1].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
    axes[1].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
    axes[1].set_title("PCA loadings — which segments drive separation")
    axes[1].axhline(0, color="gray", linewidth=0.5)
    axes[1].axvline(0, color="gray", linewidth=0.5)
    axes[1].grid(True, alpha=0.3)
    axes[1].set_aspect("equal")

    plt.suptitle("Joint 8D GMM: PCA projection", fontsize=14, weight="bold")
    plt.tight_layout()
    plt.show()


In [ ]:
# ── 14. Per-segment vs joint k comparison ─────────────────
if joint_data:
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(SEGMENT_NAMES))
    width = 0.35

    per_seg_ks = [seg_data.get(s, {}).get("primary_k", 0) for s in SEGMENT_NAMES]
    joint_k = joint_data["primary_k"]

    bars1 = ax.bar(x - width/2, per_seg_ks, width, label="Per-segment 1D GMM",
                   color="steelblue", edgecolor="black")
    bars2 = ax.bar(x + width/2, [joint_k] * len(SEGMENT_NAMES), width,
                   label=f"Joint 8D GMM (k={joint_k})", color="coral", edgecolor="black", alpha=0.7)

    ax.set_xticks(x)
    ax.set_xticklabels(SEGMENT_NAMES)
    ax.set_ylabel("Number of components")
    ax.set_title("Per-segment k vs joint 8D k", fontsize=13, weight="bold")
    ax.set_yticks(range(0, 8))
    ax.legend()
    ax.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()
    plt.show()


In [ ]:
[markdown]
# ## Summary table


In [ ]:
# ── 15. Summary table ─────────────────────────────────────
rows = []
for seg_name in SEGMENT_NAMES:
    if seg_name not in seg_data:
        continue
    sd = seg_data[seg_name]
    it = next(r for r in sd["iterations"] if r["k"] == sd["primary_k"])
    rows.append({
        "Segment": seg_name,
        "n_pairs": f"{sd['n_pairs_total']:,}",
        "k (BIC)": sd["best_k_per_criterion"]["bic"],
        "k (AIC)": sd["best_k_per_criterion"]["aic"],
        "k (ICL)": sd["best_k_per_criterion"]["icl"],
        "k (modBIC)": sd["best_k_per_criterion"]["mod_bic"],
        "Selected k": sd["primary_k"],
        "Component means": ", ".join(f"{m:.3f}" for m in sorted(it["full_data_means"])),
        "Component weights": ", ".join(f"{w:.2f}" for w in
            [it["full_data_weights"][i] for i in np.argsort(it["full_data_means"])]),
    })

summary_df = pd.DataFrame(rows)
display(summary_df)


In [ ]:
# %% ── Pairwise 2D density plots (heatmap style) ─────────────
from itertools import combinations

# Load all segment distance vectors (filtered to [0, 0.1])
print("Loading distance vectors for all segments...")
seg_vectors = {}
for seg_idx, seg_name in enumerate(SEGMENT_NAMES):
    dist_file = DIST_DIR / f"symmetric_distances_{seg_idx + 1}.parquet"
    df = pd.read_parquet(dist_file)
    mat = df.iloc[:, 1:].values.astype(float)
    n = min(mat.shape[0], mat.shape[1])
    seg_vectors[seg_name] = mat[:n, :n]

# Use the smallest matrix size across segments
n_samples = min(v.shape[0] for v in seg_vectors.values())
triu = np.triu_indices(n_samples, k=1)

# Build raw distance columns (unfiltered — we'll filter per-plot)
dist_cols = {}
for seg_name in SEGMENT_NAMES:
    dist_cols[seg_name] = seg_vectors[seg_name][:n_samples, :n_samples][triu]

print(f"Pairs per segment: {len(triu[0]):,}")


# %% ── 28-panel pairwise heatmap grid ─────────────────────────
n_seg = len(SEGMENT_NAMES)
fig, axes = plt.subplots(n_seg - 1, n_seg - 1, figsize=(3 * (n_seg - 1), 3 * (n_seg - 1)))

# Only fill lower triangle
for i in range(n_seg - 1):
    for j in range(n_seg - 1):
        ax = axes[i][j]
        if j > i:
            ax.axis("off")
            continue

        seg_x = SEGMENT_NAMES[j]
        seg_y = SEGMENT_NAMES[i + 1]
        dx = dist_cols[seg_x]
        dy = dist_cols[seg_y]

        # Filter both to [0, 0.1]
        mask = (dx >= 0) & (dx <= 0.1) & (dy >= 0) & (dy <= 0.1)
        dx_f, dy_f = dx[mask], dy[mask]

        ax.hist2d(dx_f, dy_f, bins=80, cmap="magma_r", cmin=1)
        ax.set_xlim(0, 0.1)
        ax.set_ylim(0, 0.1)
        ax.set_aspect("equal")

        if i == n_seg - 2:
            ax.set_xlabel(seg_x, fontsize=9)
        else:
            ax.set_xticklabels([])
        if j == 0:
            ax.set_ylabel(seg_y, fontsize=9)
        else:
            ax.set_yticklabels([])
        ax.tick_params(labelsize=6)

plt.suptitle("Pairwise distance distributions (2D heatmaps, filtered to [0, 0.1])",
             fontsize=14, weight="bold")
plt.tight_layout()
plt.show()


# %% ── Same grid but colored by joint 8D GMM cluster ─────────
# Load pair assignments from joint 8D GMM
pair_path = JOINT_DIR / "pair_assignments.parquet"
if pair_path.exists():
    pairs_8d = pd.read_parquet(pair_path)
    primary_k = pairs_8d["cluster"].nunique()
    cluster_colors = plt.cm.tab10(np.arange(primary_k))

    fig, axes = plt.subplots(n_seg - 1, n_seg - 1, figsize=(3 * (n_seg - 1), 3 * (n_seg - 1)))

    for i in range(n_seg - 1):
        for j in range(n_seg - 1):
            ax = axes[i][j]
            if j > i:
                ax.axis("off")
                continue

            seg_x = SEGMENT_NAMES[j]
            seg_y = SEGMENT_NAMES[i + 1]
            col_x = f"dist_{seg_x}"
            col_y = f"dist_{seg_y}"

            for c in range(primary_k):
                mask = pairs_8d["cluster"] == c
                ax.scatter(pairs_8d.loc[mask, col_x],
                           pairs_8d.loc[mask, col_y],
                           s=0.1, alpha=0.1, color=cluster_colors[c], rasterized=True)

            ax.set_xlim(0, 0.1)
            ax.set_ylim(0, 0.1)
            ax.set_aspect("equal")

            if i == n_seg - 2:
                ax.set_xlabel(seg_x, fontsize=9)
            else:
                ax.set_xticklabels([])
            if j == 0:
                ax.set_ylabel(seg_y, fontsize=9)
            else:
                ax.set_yticklabels([])
            ax.tick_params(labelsize=6)

    # Legend in the empty upper-right corner
    from matplotlib.lines import Line2D
    legend_handles = [Line2D([0], [0], marker="o", color="w", markerfacecolor=cluster_colors[c],
                              markersize=8, label=f"Cluster {c}") for c in range(primary_k)]
    axes[0][-1].legend(handles=legend_handles, loc="center", fontsize=10, frameon=False)
    axes[0][-1].axis("off")

    plt.suptitle(f"Pairwise distances colored by joint 8D GMM cluster (k={primary_k})",
                 fontsize=14, weight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("No pair_assignments.parquet found — run gmm_joint_8d.sh first")


# %% ── Focus plot: pick two segments to examine closely ───────
seg_x, seg_y = "HA", "NA"  # ← change these

dx = dist_cols[seg_x]
dy = dist_cols[seg_y]
mask = (dx >= 0) & (dx <= 0.1) & (dy >= 0) & (dy <= 0.1)
dx_f, dy_f = dx[mask], dy[mask]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: 2D heatmap
h = axes[0].hist2d(dx_f, dy_f, bins=100, cmap="magma_r", cmin=1)
plt.colorbar(h[3], ax=axes[0], label="Count")
axes[0].set_xlabel(f"{seg_x} distance")
axes[0].set_ylabel(f"{seg_y} distance")
axes[0].set_title(f"{seg_x} vs {seg_y} — density")
axes[0].set_aspect("equal")

# Panel 2: contour plot
from scipy.stats import gaussian_kde
try:
    xy = np.vstack([dx_f, dy_f])
    # Subsample for KDE speed
    if len(dx_f) > 50000:
        sub = np.random.choice(len(dx_f), 50000, replace=False)
        xy_sub = xy[:, sub]
    else:
        xy_sub = xy
    kde = gaussian_kde(xy_sub)
    xgrid = np.linspace(0, 0.1, 100)
    ygrid = np.linspace(0, 0.1, 100)
    X, Y = np.meshgrid(xgrid, ygrid)
    Z = kde(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)
    axes[1].contourf(X, Y, Z, levels=20, cmap="magma_r")
    axes[1].contour(X, Y, Z, levels=10, colors="white", linewidths=0.3)
    axes[1].set_xlabel(f"{seg_x} distance")
    axes[1].set_ylabel(f"{seg_y} distance")
    axes[1].set_title(f"{seg_x} vs {seg_y} — KDE contours")
    axes[1].set_aspect("equal")
except Exception as e:
    axes[1].text(0.5, 0.5, f"KDE failed:\n{e}", ha="center", va="center", transform=axes[1].transAxes)

# Panel 3: colored by 8D cluster (if available)
if pair_path.exists():
    col_x = f"dist_{seg_x}"
    col_y = f"dist_{seg_y}"
    for c in range(primary_k):
        m = pairs_8d["cluster"] == c
        axes[2].scatter(pairs_8d.loc[m, col_x], pairs_8d.loc[m, col_y],
                        s=0.5, alpha=0.15, color=cluster_colors[c], label=f"Cluster {c}",
                        rasterized=True)
    axes[2].legend(markerscale=20, fontsize=9)
    axes[2].set_xlabel(f"{seg_x} distance")
    axes[2].set_ylabel(f"{seg_y} distance")
    axes[2].set_title(f"{seg_x} vs {seg_y} — 8D GMM clusters")
    axes[2].set_aspect("equal")
    axes[2].set_xlim(0, 0.1)
    axes[2].set_ylim(0, 0.1)

plt.suptitle(f"Detailed pairwise view: {seg_x} vs {seg_y}", fontsize=14, weight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# %% ── Visual comparison: k=3 vs k=7 for each segment ────────
from scipy.stats import norm

fig, axes = plt.subplots(2, 8, figsize=(28, 8))

for seg_idx, seg_name in enumerate(SEGMENT_NAMES):
    if seg_name not in seg_data:
        continue
    sd = seg_data[seg_name]
    upper_tri = load_distance_vector(seg_idx)
    x = np.linspace(upper_tri.min(), upper_tri.max(), 1000)

    for row, target_k in enumerate([3, 7]):
        ax = axes[row, seg_idx]
        # Find this k in iterations
        it = next((r for r in sd["iterations"] if r["k"] == target_k), None)
        if it is None:
            ax.text(0.5, 0.5, f"k={target_k} not fitted", 
                    ha="center", va="center", transform=ax.transAxes)
            continue

        weights = it["full_data_weights"]
        means = it["full_data_means"]
        covs = it["full_data_covariances"]
        ck = plt.cm.tab10(np.arange(target_k))

        ax.hist(upper_tri, bins=150, density=True, alpha=0.3, 
                color="gray", edgecolor="none")
        for c in range(target_k):
            sigma = np.sqrt(covs[c])
            curve = weights[c] * norm.pdf(x, means[c], sigma)
            ax.plot(x, curve, color=ck[c], linewidth=1.5)

        ax.set_title(f"{seg_name} k={target_k}", fontsize=10, 
                     weight="bold")
        if seg_idx == 0:
            ax.set_ylabel("Density")
        ax.set_xlim(0, 0.1)

plt.suptitle("k=3 (top) vs k=7 (bottom): are extra components real peaks or tail-splitting?",
             fontsize=14, weight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# %% ── Project joint 8D GMM onto per-segment marginals ────────
# For a multivariate Gaussian, the marginal on dimension d is:
#   N(mu_d, Sigma_dd)
# So for each component, we just extract the d-th mean and d-th diagonal
# of the covariance matrix, then plot the weighted 1D mixture.

joint_json_path = JOINT_DIR / "gmm_joint_8d.json"
with open(joint_json_path) as f:
    joint_data = json.load(f)

primary_k = joint_data["primary_k"]
it = next(r for r in joint_data["iterations"] if r["k"] == primary_k)

weights = np.array(it["full_data_weights"])
means_8d = np.array(it["full_data_means"])       # shape (k, 8)
covs_8d = np.array(it["full_data_covariances"])   # shape (k, 8, 8)

# Undo the standardisation to get back to raw distance space
scaler_means = np.array(joint_data["scaler"]["means"])
scaler_stds = np.array(joint_data["scaler"]["stds"])

# Transform back: raw = scaled * std + mean
means_raw = means_8d * scaler_stds + scaler_means
# Variance in raw space: var_raw = var_scaled * std^2
covs_raw = covs_8d * (scaler_stds[None, :, None] * scaler_stds[None, None, :])

fig, axes = plt.subplots(2, 4, figsize=(24, 10))
axes = axes.flatten()
cluster_colors = plt.cm.tab10(np.arange(primary_k))
x = np.linspace(0, 0.1, 1000)

for d, seg_name in enumerate(SEGMENT_NAMES):
    ax = axes[d]

    # Load actual data for this segment
    upper_tri = load_distance_vector(d)
    ax.hist(upper_tri, bins=150, density=True, alpha=0.3,
            color="gray", edgecolor="none", label="Data")

    # Plot each component's marginal on this dimension
    total = np.zeros_like(x)
    for c in range(primary_k):
        mu_d = means_raw[c, d]
        sigma_d = np.sqrt(covs_raw[c, d, d])
        component = weights[c] * norm.pdf(x, mu_d, sigma_d)
        ax.plot(x, component, color=cluster_colors[c], linewidth=2,
                label=f"C{c} (μ={mu_d:.3f}, w={weights[c]:.2f})")
        total += component

    ax.plot(x, total, "k--", linewidth=2, label="Total mixture")
    ax.set_xlim(0, 0.1)
    ax.set_title(seg_name, fontsize=12, weight="bold")
    ax.set_xlabel("Distance")
    if d % 4 == 0:
        ax.set_ylabel("Density")
    ax.legend(fontsize=6, loc="upper right")

plt.suptitle(f"Joint 8D GMM (k={primary_k}): marginal projections onto each segment",
             fontsize=14, weight="bold")
plt.tight_layout()
plt.show()


# %% ── Same but comparing per-segment GMM vs joint 8D GMM ────
# Side by side: does the joint model capture the same structure
# as the independent per-segment fits?

fig, axes = plt.subplots(2, 8, figsize=(28, 8))

for d, seg_name in enumerate(SEGMENT_NAMES):
    upper_tri = load_distance_vector(d)
    x_range = np.linspace(0, 0.1, 1000)

    # ── Top row: per-segment 1D GMM ──
    ax = axes[0, d]
    ax.hist(upper_tri, bins=150, density=True, alpha=0.3,
            color="gray", edgecolor="none")

    if seg_name in seg_data:
        sd = seg_data[seg_name]
        pk = sd["primary_k"]
        it_1d = next(r for r in sd["iterations"] if r["k"] == pk)
        colors_1d = plt.cm.Set2(np.arange(pk))
        total_1d = np.zeros_like(x_range)
        for c in range(pk):
            mu = it_1d["full_data_means"][c]
            sigma = np.sqrt(it_1d["full_data_covariances"][c])
            comp = it_1d["full_data_weights"][c] * norm.pdf(x_range, mu, sigma)
            ax.plot(x_range, comp, color=colors_1d[c], linewidth=1.5)
            total_1d += comp
        ax.plot(x_range, total_1d, "k--", linewidth=1.5)
        ax.set_title(f"{seg_name} — 1D GMM (k={pk})", fontsize=10, weight="bold")
    else:
        ax.set_title(f"{seg_name} — no 1D fit", fontsize=10)
    ax.set_xlim(0, 0.1)
    if d == 0:
        ax.set_ylabel("Per-segment GMM")

    # ── Bottom row: joint 8D GMM marginal ──
    ax = axes[1, d]
    ax.hist(upper_tri, bins=150, density=True, alpha=0.3,
            color="gray", edgecolor="none")

    total_8d = np.zeros_like(x_range)
    for c in range(primary_k):
        mu_d = means_raw[c, d]
        sigma_d = np.sqrt(covs_raw[c, d, d])
        comp = weights[c] * norm.pdf(x_range, mu_d, sigma_d)
        ax.plot(x_range, comp, color=cluster_colors[c], linewidth=1.5)
        total_8d += comp
    ax.plot(x_range, total_8d, "k--", linewidth=1.5)
    ax.set_xlim(0, 0.1)
    ax.set_title(f"{seg_name} — 8D marginal (k={primary_k})", fontsize=10, weight="bold")
    if d == 0:
        ax.set_ylabel("Joint 8D marginal")

plt.suptitle("Per-segment 1D GMM (top) vs Joint 8D GMM marginal projection (bottom)",
             fontsize=14, weight="bold")
plt.tight_layout()
plt.show()


# %% ── Project onto 2D pairwise marginals ────────────────────
# Same idea but for pairs of segments: extract the 2x2 marginal
# covariance and plot the ellipses on top of the 2D heatmaps.

from matplotlib.patches import Ellipse

def plot_gmm_ellipse(ax, mean_2d, cov_2d, color, weight, n_std=2):
    """Plot a 2D Gaussian as an ellipse at n_std standard deviations."""
    eigenvalues, eigenvectors = np.linalg.eigh(cov_2d)
    angle = np.degrees(np.arctan2(eigenvectors[1, 0], eigenvectors[0, 0]))
    width, height = 2 * n_std * np.sqrt(eigenvalues)
    ellipse = Ellipse(xy=mean_2d, width=width, height=height, angle=angle,
                      facecolor=color, alpha=0.15 * min(weight * 5, 1),
                      edgecolor=color, linewidth=2)
    ax.add_patch(ellipse)
    ax.plot(*mean_2d, "x", color=color, markersize=8, markeredgewidth=2)

# Pick segment pairs to visualise (change these)
pairs_to_show = [("PB2", "HA"), ("HA", "NA"), ("NP", "HA"),
                 ("PB1", "PA"), ("NA", "MP"), ("NS", "NP")]

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

for ax, (seg_x, seg_y) in zip(axes, pairs_to_show):
    dx_idx = SEGMENT_NAMES.index(seg_x)
    dy_idx = SEGMENT_NAMES.index(seg_y)

    dx = dist_cols[seg_x]
    dy = dist_cols[seg_y]
    mask = (dx >= 0) & (dx <= 0.1) & (dy >= 0) & (dy <= 0.1)

    ax.hist2d(dx[mask], dy[mask], bins=80, cmap="magma_r", cmin=1)

    # Overlay GMM component ellipses
    for c in range(primary_k):
        mean_2d = means_raw[c, [dx_idx, dy_idx]]
        cov_2d = covs_raw[c, np.ix_([dx_idx, dy_idx], [dx_idx, dy_idx])]
        plot_gmm_ellipse(ax, mean_2d, cov_2d, cluster_colors[c], weights[c])

    ax.set_xlim(0, 0.1)
    ax.set_ylim(0, 0.1)
    ax.set_xlabel(f"{seg_x} distance")
    ax.set_ylabel(f"{seg_y} distance")
    ax.set_title(f"{seg_x} vs {seg_y}", fontsize=12, weight="bold")
    ax.set_aspect("equal")

plt.suptitle(f"Joint 8D GMM (k={primary_k}): 2D marginal ellipses on pairwise heatmaps",
             fontsize=14, weight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# %% ── Compare all k marginal projections to find the right k ─
joint_json_path = JOINT_DIR / "gmm_joint_8d.json"
with open(joint_json_path) as f:
    joint_data = json.load(f)

scaler_means = np.array(joint_data["scaler"]["means"])
scaler_stds = np.array(joint_data["scaler"]["stds"])

fig, axes = plt.subplots(6, 8, figsize=(28, 22))  # rows=k values, cols=segments

for row_idx, target_k in enumerate(range(2, 8)):
    it = next((r for r in joint_data["iterations"] if r["k"] == target_k), None)
    if it is None:
        continue

    weights = np.array(it["full_data_weights"])
    means_8d = np.array(it["full_data_means"])
    covs_8d = np.array(it["full_data_covariances"])

    # Unscale to raw distances
    means_raw = means_8d * scaler_stds + scaler_means
    covs_raw = covs_8d * (scaler_stds[None, :, None] * scaler_stds[None, None, :])

    for d, seg_name in enumerate(SEGMENT_NAMES):
        ax = axes[row_idx, d]
        upper_tri = load_distance_vector(d)
        x = np.linspace(0, 0.1, 500)

        ax.hist(upper_tri, bins=100, density=True, alpha=0.3,
                color="gray", edgecolor="none")

        total = np.zeros_like(x)
        colors_k = plt.cm.tab10(np.arange(target_k))
        for c in range(target_k):
            mu_d = means_raw[c, d]
            sigma_d = np.sqrt(covs_raw[c, d, d])
            comp = weights[c] * norm.pdf(x, mu_d, sigma_d)
            ax.plot(x, comp, color=colors_k[c], linewidth=1.2)
            total += comp

        ax.plot(x, total, "k--", linewidth=1.5)
        ax.set_xlim(0, 0.1)
        ax.tick_params(labelsize=6)

        if row_idx == 0:
            ax.set_title(seg_name, fontsize=23, weight="bold")
        if d == 0:
            ax.set_ylabel(f"k={target_k}", fontsize=23, weight="bold")
        if row_idx < 5:
            ax.set_xticklabels([])

plt.suptitle("Joint 8D GMM: marginal projections for k=2 through k=7",
             "\n \n ",
             fontsize=30, weight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# %% ── The right way to evaluate the joint model ─────────────
# Don't judge by marginal fit — judge by cluster coherence.
# Use k=3 or k=4 diag and look at:
# 1. Do the clusters separate cleanly in the 2D heatmaps?
# 2. Do the cluster mean profiles make biological sense?
# 3. Do discordant pairs (different cluster in HA vs PB1) correspond 
#    to known reassortment events?

from sklearn.preprocessing import StandardScaler

# Load the diag results
JOINT_DIAG_DIR = BASE_DIR / "famsa_per_segment_analysis" / "gmm_joint_8d_diag"
with open(JOINT_DIAG_DIR / "gmm_joint_8d_diag.json") as f:
    diag_data = json.load(f)

# Force k=3 regardless of what ICL picked
TARGET_K = 3  # ← try 3 and 4

it = next(r for r in diag_data["iterations"] if r["k"] == TARGET_K)
scaler_means = np.array(diag_data["scaler"]["means"])
scaler_stds = np.array(diag_data["scaler"]["stds"])

means_raw = np.array(it["full_data_means"]) * scaler_stds + scaler_means
vars_raw = np.array(it["full_data_covariances"]) * scaler_stds**2
weights = np.array(it["full_data_weights"])

print(f"k={TARGET_K} diag model:")
for c in range(TARGET_K):
    print(f"  Cluster {c} (w={weights[c]:.2f}): means = {np.round(means_raw[c], 4)}")


# %% ── Cluster mean profile heatmap (the key plot) ───────────
fig, ax = plt.subplots(figsize=(10, max(3, TARGET_K + 1)))
im = ax.imshow(means_raw, aspect="auto", cmap="viridis")
ax.set_xticks(range(8))
ax.set_xticklabels(SEGMENT_NAMES, fontsize=23)
ax.set_yticks(range(TARGET_K))
ax.set_yticklabels([f"Cluster {c} (w={weights[c]:.2f})" for c in range(TARGET_K)], fontsize=10)
for i in range(TARGET_K):
    for j in range(8):
        ax.text(j, i, f"{means_raw[i, j]:.3f}", ha="center", va="center",
                fontsize=20, color="white" if means_raw[i, j] > means_raw.mean() else "black")
plt.colorbar(im, label="Mean distance", shrink=0.8)
ax.set_title(f"Joint 8D GMM (diag, k={TARGET_K}): what each cluster means",
             fontsize=30, weight="bold")
plt.tight_layout()
plt.show()

# Interpretation:
# - A cluster with LOW mean across all segments = pairs within the same lineage
# - A cluster with HIGH mean across all segments = pairs between distant lineages
# - A cluster with MIXED means (low in some, high in others) = REASSORTMENT signal


# %% ── Reconstruct labels for k=TARGET_K and plot 2D heatmaps ─
from sklearn.mixture import GaussianMixture

# Reload pair data
pair_path = JOINT_DIAG_DIR / "pair_assignments.parquet"
pairs = pd.read_parquet(pair_path)
dist_cols_names = [f"dist_{s}" for s in SEGMENT_NAMES]
feature_matrix = pairs[dist_cols_names].values

scaler = StandardScaler()
feature_matrix_scaled = scaler.fit_transform(feature_matrix)

# Refit with target k (since the saved labels are for ICL-selected k)
gmm = GaussianMixture(n_components=TARGET_K, covariance_type='diag',
                       random_state=42, n_init=5, max_iter=500)
gmm.fit(feature_matrix_scaled)
labels = gmm.predict(feature_matrix_scaled)

print(f"Cluster sizes: {np.bincount(labels)}")

# ── Pairwise heatmap grid colored by cluster ──
n_seg = len(SEGMENT_NAMES)
cluster_colors = plt.cm.tab10(np.arange(TARGET_K))

fig, axes = plt.subplots(n_seg - 1, n_seg - 1, figsize=(3 * (n_seg - 1), 3 * (n_seg - 1)))

for i in range(n_seg - 1):
    for j in range(n_seg - 1):
        ax = axes[i][j]
        if j > i:
            ax.axis("off")
            continue
        seg_x = SEGMENT_NAMES[j]
        seg_y = SEGMENT_NAMES[i + 1]
        col_x = f"dist_{seg_x}"
        col_y = f"dist_{seg_y}"

        # Subsample for plotting speed
        n_plot = min(100_000, len(pairs))
        plot_idx = np.random.choice(len(pairs), n_plot, replace=False)

        for c in range(TARGET_K):
            m = labels[plot_idx] == c
            ax.scatter(pairs[col_x].values[plot_idx[m]],
                       pairs[col_y].values[plot_idx[m]],
                       s=0.3, alpha=0.15, color=cluster_colors[c], rasterized=True)
        ax.set_xlim(0, 0.1)
        ax.set_ylim(0, 0.1)
        ax.set_aspect("equal")
        if i == n_seg - 2:
            ax.set_xlabel(seg_x, fontsize=20)
        else:
            ax.set_xticklabels([])
        if j == 0:
            ax.set_ylabel(seg_y, fontsize=20)
        else:
            ax.set_yticklabels([])
        ax.tick_params(labelsize=6)

from matplotlib.lines import Line2D
legend_handles = [Line2D([0], [0], marker="o", color="w", markerfacecolor=cluster_colors[c],
                          markersize=8, label=f"Cluster {c} (w={weights[c]:.2f})")
                  for c in range(TARGET_K)]
axes[0][-1].legend(handles=legend_handles, loc="center", fontsize=25, frameon=False)
axes[0][-1].axis("off")

plt.suptitle(f"Joint 8D GMM (diag, k={TARGET_K}): pairwise distances colored by cluster",
             fontsize=20, weight="bold")
plt.tight_layout()
plt.show()